# 🚀 Phiên Bản LIÊM (LGB v6 - Fixed)
Notebook này tích hợp toàn bộ source code của v6.

**Các fix so với bản gốc:**
- Uncomment `import os` và `import time`
- Xóa orphaned import fragment trong `models.py` section
- Thay thế MAIN section dùng `train_lgbm/predict_lgbm` (undefined) bằng `train_and_predict_all_horizons()` (đã có đủ code)


In [ ]:

import pandas as pd
import numpy as np
import gc
import warnings
import lightgbm as lgb
import time
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

# ========================================
# config.py
# ========================================

import os

# ── Environment Detection ──
IS_KAGGLE = os.path.exists('/kaggle/input')

# ── Data Paths ──
if IS_KAGGLE:
    TRAIN_PATH = "/kaggle/input/competitions/ts-forecasting/train.parquet"
    TEST_PATH  = "/kaggle/input/competitions/ts-forecasting/test.parquet"
    OUTPUT_DIR = "/kaggle/working"
else:
    TRAIN_PATH = "data/train.parquet"
    TEST_PATH  = "data/test.parquet"
    OUTPUT_DIR = "submissions"

# ── Constants ──
HORIZONS = [1, 3, 10, 25]
TARGET = "y_target"
WEIGHT = "weight"
VAL_THRESHOLD = 3500

# ── Column Groups ──
CAT_COLS   = ["code", "sub_code", "sub_category"]
META_COLS  = ["id", "code", "sub_code", "sub_category", "horizon", "ts_index"]
GROUP_COLS = ["code", "sub_code", "sub_category", "horizon"]

# ── Feature Engineering Config ──
KEY_FEATURES = ["feature_al", "feature_am", "feature_cg", "feature_by"]
EXTRA_FEATURES = ["feature_s"]
LAG_STEPS = [1, 3, 5, 10, 25]
ROLLING_WINDOWS = [5, 10, 20]

# ── Seeds ──
N_SEEDS = 15
SEEDS = [42, 2024, 12345, 99, 420, 777, 1337, 2025, 7, 11, 314, 617, 888, 999, 5555]

# ── LightGBM Params ──
LGB_PARAMS = {
    "objective":        "regression",
    "metric":           "rmse",
    "boosting_type":    "gbdt",
    "learning_rate":    0.015,
    "num_leaves":       127,
    "max_depth":        -1,
    "min_child_samples": 200,
    "feature_fraction": 0.60,
    "bagging_fraction": 0.75,
    "bagging_freq":     5,
    "lambda_l1":        2.0,
    "lambda_l2":        10.0,
    "extra_trees":      True,
    "path_smooth":      1.0,
    "verbosity":        -1,
    "n_jobs":           -1,
}


# ========================================
# utils.py
# ========================================

# import pandas as pd
# import numpy as np
# import lightgbm as lgb
import matplotlib.pyplot as plt


def plot_feature_importance(model, top_n=30, figsize=(10, 12)):
    importance = pd.DataFrame({
        "feature": model.feature_name(),
        "importance": model.feature_importance(importance_type="gain"),
    }).sort_values("importance", ascending=False)
    
    fig, ax = plt.subplots(figsize=figsize)
    top = importance.head(top_n)
    ax.barh(range(len(top)), top["importance"].values)
    ax.set_yticks(range(len(top)))
    ax.set_yticklabels(top["feature"].values)
    ax.invert_yaxis()
    ax.set_title(f"Top {top_n} Feature Importance (Gain)")
    ax.set_xlabel("Importance (Gain)")
    plt.tight_layout()
    plt.show()
    
    return importance


def print_data_summary(df, name="DataFrame"):
    print(f"\n{'='*50}")
    print(f"Summary: {name}")
    print(f"{'='*50}")
    print(f"Shape: {df.shape}")
    print(f"Memory: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
    
    if "ts_index" in df.columns:
        print(f"ts_index range: {df['ts_index'].min()} → {df['ts_index'].max()}")
    
    if "horizon" in df.columns:
        print(f"Horizons: {sorted(df['horizon'].unique())}")
    
    for col in ["code", "sub_code", "sub_category"]:
        if col in df.columns:
            print(f"{col}: {df[col].nunique()} unique values")
    
    # Missing values
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if len(missing) > 0:
        print(f"\nMissing values ({len(missing)} columns):")
        for col, count in missing.items():
            pct = 100 * count / len(df)
            print(f"  {col}: {count:,} ({pct:.1f}%)")
    else:
        print("\nNo missing values!")


def check_submission(submission_df, test_df):
    errors = []
    
    # Check columns
    if list(submission_df.columns) != ["id", "prediction"]:
        errors.append(f"Expected columns ['id', 'prediction'], got {list(submission_df.columns)}")
    
    # Check row count
    if len(submission_df) != len(test_df):
        errors.append(f"Row count: {len(submission_df)} vs expected {len(test_df)}")
    
    # Check NaN
    nan_count = submission_df["prediction"].isnull().sum()
    if nan_count > 0:
        errors.append(f"Found {nan_count} NaN predictions")
    
    # Check ID match
    if set(submission_df["id"]) != set(test_df["id"]):
        missing = set(test_df["id"]) - set(submission_df["id"])
        extra   = set(submission_df["id"]) - set(test_df["id"])
        if missing:
            errors.append(f"{len(missing)} missing IDs")
        if extra:
            errors.append(f"{len(extra)} extra IDs")
    
    if errors:
        print("SUBMISSION VALIDATION FAILED:")
        for e in errors:
            print(f"  - {e}")
        return False
    else:
        print("Submission looks valid!")
        print(f"  Rows: {len(submission_df):,}")
        print(f"  Prediction range: [{submission_df['prediction'].min():.4f}, "
              f"{submission_df['prediction'].max():.4f}]")
        print(f"  Prediction mean: {submission_df['prediction'].mean():.4f}")
        return True


# ========================================
# data_loader.py
# ========================================

# import pandas as pd
# import numpy as np


def reduce_mem_usage(df, verbose=True):
    start_mem = df.memory_usage(deep=True).sum() / 1024**2
    
    for col in df.columns:
        col_type = df[col].dtype
        if col_type == object or str(col_type) == "category":
            continue
        
        c_min, c_max = df[col].min(), df[col].max()
        
        if str(col_type).startswith("int"):
            if c_min >= np.iinfo(np.int8).min and c_max <= np.iinfo(np.int8).max:
                df[col] = df[col].astype(np.int8)
            elif c_min >= np.iinfo(np.int16).min and c_max <= np.iinfo(np.int16).max:
                df[col] = df[col].astype(np.int16)
            elif c_min >= np.iinfo(np.int32).min and c_max <= np.iinfo(np.int32).max:
                df[col] = df[col].astype(np.int32)
        else:
            df[col] = df[col].astype(np.float32)
    
    end_mem = df.memory_usage(deep=True).sum() / 1024**2
    if verbose:
        print(f"Memory: {start_mem:.1f}MB → {end_mem:.1f}MB "
              f"({100 * (start_mem - end_mem) / start_mem:.1f}% reduction)")
    return df


def load_data(reduce_memory=True):
    print("Loading data...")
    train_df = pd.read_parquet(TRAIN_PATH)
    test_df  = pd.read_parquet(TEST_PATH)
    print(f"  Train: {train_df.shape}, Test: {test_df.shape}")
    
    #  1. Native categorical 
    for col in CAT_COLS:
        # Combine categories so train & test share the same category list
        all_cats = pd.CategoricalDtype(
            categories=sorted(set(train_df[col].unique()) | set(test_df[col].unique()))
        )
        train_df[col] = train_df[col].astype(all_cats)
        test_df[col]  = test_df[col].astype(all_cats)
    
    #  2. Sort by group + time 
    sort_cols = GROUP_COLS + ["ts_index"]
    train_df = train_df.sort_values(sort_cols).reset_index(drop=True)
    test_df  = test_df.sort_values(sort_cols).reset_index(drop=True)
    
    #  3. Forward fill missing values 
    feature_cols = [c for c in train_df.columns if c.startswith("feature_")]
    
    for col in feature_cols:
        if train_df[col].isnull().any():
            # Forward fill within each entity-horizon group
            train_df[col] = train_df.groupby(GROUP_COLS)[col].ffill()
            # Fallback: fill remaining NaN with column median (from train)
            median_val = train_df[col].median()
            train_df[col] = train_df[col].fillna(median_val)
            
            if col in test_df.columns and test_df[col].isnull().any():
                test_df[col] = test_df.groupby(GROUP_COLS)[col].ffill()
                test_df[col] = test_df[col].fillna(median_val)  # Use train median
    
    #  4. Reduce memory 
    if reduce_memory:
        print("Reducing memory...")
        train_df = reduce_mem_usage(train_df, verbose=True)
        test_df  = reduce_mem_usage(test_df, verbose=True)
    
    print("Data loading complete!")
    return train_df, test_df


def get_feature_columns(df):
    exclude = {"id", "y_target", "weight"}
    return [c for c in df.columns if c not in exclude]


def time_split(df, split_ts_index=3000):
    train_part = df[df["ts_index"] <= split_ts_index].copy()
    val_part   = df[df["ts_index"] >  split_ts_index].copy()
    print(f"Time split at ts_index={split_ts_index}:")
    print(f"  Train: {train_part.shape}, Val: {val_part.shape}")
    return train_part, val_part


# ========================================
# features.py
# ========================================

# import pandas as pd
# import numpy as np
# import gc



def compute_target_stats(df):
    stats = {
        'code': df.groupby('code')['y_target'].mean().to_dict(),
        'sub_category': df.groupby('sub_category')['y_target'].mean().to_dict(),
        'sub_code': df.groupby('sub_code')['y_target'].mean().to_dict(),
        'global_mean': float(df['y_target'].mean()),
        'sub_code_q10': df.groupby('sub_code')['y_target'].quantile(0.10).to_dict(),
        'sub_code_q90': df.groupby('sub_code')['y_target'].quantile(0.90).to_dict(),
        'sub_code_std': df.groupby('sub_code')['y_target'].std().fillna(1.0).to_dict(),
        'sub_cat_std': df.groupby('sub_category')['y_target'].std().fillna(1.0).to_dict(),
    }
    return stats


def compute_freq_encoding(df):
    freq = {}
    for c in ['code', 'sub_code', 'sub_category']:
        if c in df.columns:
            freq[c] = df[c].value_counts(normalize=True).to_dict()
    return freq


def build_features(data, target_stats, freq_stats):
    df = data.copy()
    gm = target_stats.get('global_mean', 0.0)
    raw_cols = [c for c in df.columns if c.startswith('feature_')]

    # ── 1. Row-wise meta-features (numpy for speed) ──
    if raw_cols:
        import numpy as np
        X = df[raw_cols].fillna(0.0).values
        df['feat_mean']     = np.mean(X, axis=1).astype(np.float32)
        df['feat_std']      = np.std(X, axis=1).astype(np.float32)
        df['feat_range']    = (np.max(X, axis=1) - np.min(X, axis=1)).astype(np.float32)
        df['feat_pos_frac'] = (X > 0).mean(axis=1).astype(np.float32)
        df['feat_l2']       = np.sqrt((X ** 2).sum(axis=1)).astype(np.float32)
        del X

    # ── 2. Feature Interactions (expanded) ──
    E = lambda f: f in df.columns
    if E('feature_al') and E('feature_am'):
        df['d_al_am'] = df['feature_al'] - df['feature_am']
        df['r_al_am'] = df['feature_al'] / (df['feature_am'].abs() + 1e-7)
        df['p_al_am'] = df['feature_al'] * df['feature_am']
    if E('feature_cg') and E('feature_by'):
        df['d_cg_by'] = df['feature_cg'] - df['feature_by']
        df['r_cg_by'] = df['feature_cg'] / (df['feature_by'].abs() + 1e-7)
        df['mean_cg_by'] = (df['feature_cg'] + df['feature_by']) / 2.0
    if E('feature_al') and E('feature_bp'):
        df['d_al_bp'] = df['feature_al'] - df['feature_bp']
        df['mean_al_bp'] = (df['feature_al'] + df['feature_bp']) / 2.0
        df['r_al_bp'] = df['feature_al'] / (df['feature_bp'].abs() + 1e-7)
    if E('feature_s') and E('feature_t'):
        df['d_s_t'] = df['feature_s'] - df['feature_t']
    if E('feature_s'):
        for f in ['feature_al', 'feature_am', 'feature_cg']:
            if E(f):
                df[f's_{f.split("_")[1]}_prod'] = df['feature_s'] * df[f]
    if E('feature_am') and E('feature_bz'):
        df['p_am_bz'] = df['feature_am'] * df['feature_bz']
    if E('feature_al') and E('feature_cg'):
        df['al_x_cg'] = df['feature_al'] * df['feature_cg']
    if E('feature_a') and E('feature_b'):
        df['d_a_b'] = df['feature_a'] - df['feature_b']
    if E('feature_c') and E('feature_d'):
        df['d_c_d'] = df['feature_c'] - df['feature_d']
    if E('feature_e') and E('feature_f'):
        df['d_e_f'] = df['feature_e'] - df['feature_f']
    if all(E(c) for c in ['feature_al', 'feature_bp', 'feature_am', 'feature_bq']):
        df['wap'] = (df['feature_al'] * df['feature_bq'] + df['feature_bp'] * df['feature_am']) / (df['feature_am'] + df['feature_bq'] + 1e-7)

    # ── 3. Target Encoding (from train stats) ──
    for c in ['code', 'sub_category', 'sub_code']:
        if c in target_stats:
            df[c + '_enc'] = df[c].map(target_stats[c]).fillna(gm).astype(np.float32)
    df['sc_q10']    = df['sub_code'].map(target_stats.get('sub_code_q10', {})).fillna(gm).astype(np.float32)
    df['sc_q90']    = df['sub_code'].map(target_stats.get('sub_code_q90', {})).fillna(gm).astype(np.float32)
    df['sc_qrange'] = (df['sc_q90'] - df['sc_q10']).astype(np.float32)
    df['sc_std']    = df['sub_code'].map(target_stats.get('sub_code_std', {})).fillna(1.0).astype(np.float32)
    df['scat_std']  = df['sub_category'].map(target_stats.get('sub_cat_std', {})).fillna(1.0).astype(np.float32)
    if 'sub_code_enc' in df.columns:
        df['code_snr']  = (df['sub_code_enc'].abs() / (df['sc_std'] + 1e-7)).astype(np.float32)

    # ── 4. Frequency Encoding (rule-safe, no target) ──
    for c in ['code', 'sub_code', 'sub_category']:
        if c in freq_stats:
            df[c + '_freq'] = df[c].map(freq_stats[c]).fillna(0).astype(np.float32)
    if 'sub_code_freq' in df.columns:
        df['sc_log_freq'] = np.log1p(df['sub_code_freq']).astype(np.float32)

    # ── 5. Cross-sectional normalization + rank ──
    cs_cols = [c for c in ['feature_al', 'feature_am', 'feature_cg', 'feature_by', 'd_al_am', 'd_cg_by', 'feat_mean'] if E(c)]
    for col in cs_cols:
        g = df.groupby('ts_index')[col]
        df[col + '_cs'] = ((df[col] - g.transform('mean')) / (g.transform('std') + 1e-7)).astype(np.float32)
        df[col + '_rank'] = g.rank(pct=True).astype(np.float32)
    if E('feature_s'):
        df['feature_s_rank'] = df.groupby('ts_index')['feature_s'].rank(pct=True).astype(np.float32)
    for f in ['feature_al', 'feature_am', 'd_al_am']:
        if E(f):
            g2 = df.groupby(['ts_index', 'sub_category'])[f]
            df[f + '_gcs'] = ((df[f] - g2.transform('mean')) / (g2.transform('std') + 1e-7)).astype(np.float32)

    # ── 6. Time features ──
    ts = df['ts_index']
    for p in [2, 3, 5, 7, 12, 24, 30]:
        df[f'ts_mod_{p}'] = (ts % p).astype(np.int8)
    df['t_sin']  = np.sin(2 * np.pi * ts / 100).astype(np.float32)
    df['t_cos']  = np.cos(2 * np.pi * ts / 100).astype(np.float32)
    df['t_sin2'] = np.sin(2 * np.pi * ts / 52).astype(np.float32)
    df['t_cos2'] = np.cos(2 * np.pi * ts / 52).astype(np.float32)
    df['ts_norm'] = (ts / 4000.0).astype(np.float32)
    if 'horizon' in df.columns:
        df['horizon_log'] = np.log1p(df['horizon']).astype(np.float32)

    # ── 7. Lifecycle ──
    df = df.sort_values(GROUP_COLS + ['ts_index'])
    df['obs_idx'] = df.groupby(GROUP_COLS).cumcount().astype(np.int32)
    first_t = df.groupby(GROUP_COLS)['ts_index'].transform('min')
    df['time_since_start'] = (df['ts_index'] - first_t).astype(np.int32)
    del first_t

    # ── 8. Lags, Rolling, EWM, Diff, Rank (WITH DATA LEAK -> NO SHIFT) ──
    key_present = [f for f in KEY_FEATURES if E(f)]
    extra_present = [f for f in EXTRA_FEATURES if E(f)]
    inter_feats = [f for f in ['d_al_am', 'd_cg_by', 'd_s_t', 'mean_al_bp', 'mean_cg_by', 'd_al_bp', 'wap'] if E(f)]
    
    grouped = df.groupby(GROUP_COLS, sort=False)
    new = {}

    for col in key_present:
        for lag in LAG_STEPS:
            new[f'{col}_lag{lag}'] = grouped[col].shift(lag).astype(np.float32)
        for w in ROLLING_WINDOWS:
            # BỎ THUỐC ĐỘC: Không shift(1), dùng luôn t hiện tại
            shifted = grouped[col]
            new[f'{col}_rmean_{w}'] = shifted.rolling(w, min_periods=1).mean().values.astype(np.float32)
            new[f'{col}_rstd_{w}']  = shifted.rolling(w, min_periods=1).std().values.astype(np.float32)
        new[f'{col}_ewm10'] = grouped[col].ewm(span=10, min_periods=1).mean().values.astype(np.float32) # BỎ THUỐC ĐỘC
        new[f'{col}_diff1'] = grouped[col].diff(1).astype(np.float32)

    for col in extra_present:
        for lag in [1, 3, 5]:
            new[f'{col}_lag{lag}'] = grouped[col].shift(lag).astype(np.float32)
        new[f'{col}_diff1'] = grouped[col].diff(1).astype(np.float32)
        # BỎ THUỐC ĐỘC 
        shifted = grouped[col]
        new[f'{col}_rmean_5'] = shifted.rolling(5, min_periods=1).mean().values.astype(np.float32)
        new[f'{col}_ewm10']   = grouped[col].ewm(span=10, min_periods=1).mean().values.astype(np.float32)

    for col in inter_feats:
        for lag in [1, 3]:
            new[f'{col}_lag{lag}'] = grouped[col].shift(lag).astype(np.float32)
        new[f'{col}_diff1'] = grouped[col].diff(1).astype(np.float32)

    if new:
        df = pd.concat([df, pd.DataFrame(new, index=df.index)], axis=1)

    # ── 9. Momentum ──
    for col in key_present:
        l1, l5, rm5 = f'{col}_lag1', f'{col}_lag5', f'{col}_rmean_5'
        if E(l1) and E(l5):
            df[f'{col}_mom15'] = (df[l1] - df[l5]).astype(np.float32)
        if E(l1) and E(rm5):
            df[f'{col}_dev5']  = (df[l1] - df[rm5]).astype(np.float32)

    # ── 10. Fill NaN/Inf ──
    preserved_y = df['y_target'].copy() if 'y_target' in df.columns else None
    preserved_w = df['weight'].copy() if 'weight' in df.columns else None
    
    df = df.fillna(0.0).replace([np.inf, -np.inf], 0.0)
    
    if preserved_y is not None: df['y_target'] = preserved_y
    if preserved_w is not None: df['weight'] = preserved_w

    import gc
    gc.collect()
    return df


def get_feature_columns(df):
    exclude = {'id', 'code', 'sub_code', 'sub_category', 'horizon', 'ts_index', 'weight', 'y_target'}
    return [c for c in df.columns if c not in exclude]


# ========================================
# models.py
# ========================================

# import lightgbm as lgb
# import numpy as np
# import pandas as pd
# import gc
# import time
from sklearn.linear_model import LinearRegression






def solve_horizon(horizon):
    t0 = time.time()
    print(f'\n{"="*60}')
    print(f'HORIZON {horizon}')
    print(f'{"="*60}')

    # ── Load data ──
    tr = pd.read_parquet(TRAIN_PATH).query('horizon == @horizon').reset_index(drop=True)
    te = pd.read_parquet(TEST_PATH).query('horizon == @horizon').reset_index(drop=True)
    print(f'Data: train={len(tr):,}, test={len(te):,}')

    # ── Stats from train only ──
    target_stats = compute_target_stats(tr)

    # ── Concat train+test for continuous lags ──
    te_tmp = te.copy()
    te_tmp[TARGET] = np.nan
    combined = pd.concat([tr, te_tmp], ignore_index=True)
    combined = combined.sort_values(GROUP_COLS + ['ts_index']).reset_index(drop=True)

    # Freq encoding on combined data (no target used → rule-safe)
    freq_stats = compute_freq_encoding(combined)

    print('Building features...')
    combined_feat = build_features(combined, target_stats, freq_stats)

    is_train = combined_feat[TARGET].notna()
    all_feat = combined_feat[is_train].reset_index(drop=True)
    te_feat = combined_feat[~is_train].reset_index(drop=True)
    print(f'Split: train_feat={len(all_feat):,}, test_feat={len(te_feat):,}')
    del combined, combined_feat, te_tmp
    gc.collect()

    feats = get_feature_columns(all_feat)
    for c in feats:
        if c not in te_feat.columns:
            te_feat[c] = 0.0
    print(f'Features: {len(feats)}')

    # ── Train/Val split ──
    tr_mask = all_feat['ts_index'] <= VAL_THRESHOLD
    va_mask = all_feat['ts_index'] > VAL_THRESHOLD

    X_tr = all_feat.loc[tr_mask, feats]
    y_tr = all_feat.loc[tr_mask, TARGET]
    w_tr = all_feat.loc[tr_mask, WEIGHT]
    X_va = all_feat.loc[va_mask, feats]
    y_va = all_feat.loc[va_mask, TARGET]
    w_va = all_feat.loc[va_mask, WEIGHT]
    print(f'Train: {len(X_tr):,}, Val: {len(X_va):,}')

    # ── Find best iteration ──
    probe = lgb.LGBMRegressor(**LGB_PARAMS, n_estimators=5000, random_state=42)
    probe.fit(
        X_tr, y_tr, sample_weight=w_tr,
        eval_set=[(X_va, y_va)], eval_sample_weight=[w_va],
        callbacks=[lgb.early_stopping(200, verbose=False)]
    )
    best_iter = probe.best_iteration_ if probe.best_iteration_ else 3000
    best_iter = max(best_iter, 50)

    # ── Validation score ──
    val_pred = probe.predict(X_va)
    val_score_raw = weighted_rmse_score(y_va.values, val_pred, w_va.values)
    print(f'Val score (probe): {val_score_raw:.6f}, best_iter={best_iter}')
    del probe
    gc.collect()

    # ── Calibration on validation ──
    calibrator = LinearRegression()
    calibrator.fit(val_pred.reshape(-1, 1), y_va.values, sample_weight=w_va.values)
    cal_val = calibrator.predict(val_pred.reshape(-1, 1)).ravel()
    val_score_cal = weighted_rmse_score(y_va.values, cal_val, w_va.values)
    print(f'Calibration: a={calibrator.coef_[0]:.4f}, b={calibrator.intercept_:.6f}')
    print(f'Val score (calibrated): {val_score_cal:.6f}')

    # ── Retrain on ALL data with N seeds ──
    X_all = all_feat[feats]
    y_all = all_feat[TARGET]
    w_all = all_feat[WEIGHT]

    test_pred = np.zeros(len(te_feat), dtype=np.float64)
    for i, seed in enumerate(SEEDS):
        if i == 0 or (i + 1) % 5 == 0:
            print(f'  Seed {i+1}/{N_SEEDS}...')
        mdl = lgb.LGBMRegressor(**{**LGB_PARAMS, 'n_estimators': best_iter}, random_state=seed)
        mdl.fit(X_all, y_all, sample_weight=w_all)
        test_pred += mdl.predict(te_feat[feats]) / N_SEEDS
        del mdl
        gc.collect()

    # ── Apply calibration to test predictions ──
    test_pred_cal = calibrator.predict(test_pred.reshape(-1, 1)).ravel()

    # ── Post-processing: percentile clip ──
    clip_lo = np.percentile(y_all.values, 0.5)
    clip_hi = np.percentile(y_all.values, 99.5)
    test_pred_clip = np.clip(test_pred_cal, clip_lo, clip_hi)
    print(f'Clip range: [{clip_lo:.2f}, {clip_hi:.2f}]')

    elapsed = (time.time() - t0) / 60
    print(f'Horizon {horizon} done in {elapsed:.1f} min')

    test_ids = te_feat['id'].values

    del all_feat, te_feat, X_tr, X_va, X_all
    gc.collect()

    return {
        'horizon': horizon,
        'ids': test_ids,
        'pred_raw': test_pred,
        'pred_clip': test_pred_clip,
        'val_score_raw': val_score_raw,
        'val_score_cal': val_score_cal,
        'val_y': y_va.values,
        'val_w': w_va.values,
        'val_pred': cal_val,
    }


def train_and_predict_all_horizons():
    print('=' * 60)
    print('V6 HYBRID — LGB + Calibration + Concat + FreqEnc')
    print('=' * 60)
    print(f'Config: {N_SEEDS} seeds, val_split={VAL_THRESHOLD}')
    print(f'Horizons: {HORIZONS}')

    results = []
    for h in HORIZONS:
        results.append(solve_horizon(h))

    # ── Build submissions ──
    sub_parts = []
    sub_raw_parts = []
    for r in results:
        sub_parts.append(pd.DataFrame({'id': r['ids'], 'prediction': r['pred_clip']}))
        sub_raw_parts.append(pd.DataFrame({'id': r['ids'], 'prediction': r['pred_raw']}))

    sub = pd.concat(sub_parts, ignore_index=True)
    sub_raw = pd.concat(sub_raw_parts, ignore_index=True)

    # ── Aggregate score ──
    all_y = np.concatenate([r['val_y'] for r in results])
    all_w = np.concatenate([r['val_w'] for r in results])
    all_p = np.concatenate([r['val_pred'] for r in results])
    agg_score = weighted_rmse_score(all_y, all_p, all_w)

    print('\n' + '=' * 60)
    print('RESULTS')
    print(f'{"H":>4} | {"Raw":>10} | {"Calibrated":>10}')
    print('-' * 30)
    for r in sorted(results, key=lambda x: x['horizon']):
        print(f"  {r['horizon']:>2} | {r['val_score_raw']:.6f} | {r['val_score_cal']:.6f}")
    print('-' * 30)
    print(f'  Aggregate (calibrated): {agg_score:.6f}')
    print('=' * 60)

    return sub, sub_raw, {r['horizon']: r['val_score_cal'] for r in results}


def create_submission(submission_df, filename="submission.csv"):
    from config import OUTPUT_DIR
    import os

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    path = os.path.join(OUTPUT_DIR, filename)

    submission_df[["id", "prediction"]].to_csv(path, index=False)
    print(f"Saved: {path} ({submission_df.shape[0]:,} rows)")
    print(submission_df.head())
    return path


# ========================================
# evaluation.py
# ========================================

# import numpy as np


def _clip01(x: float) -> float:
    return float(np.minimum(np.maximum(x, 0.0), 1.0))


def weighted_rmse_score(y_target, y_pred, w) -> float:
    y_target = np.asarray(y_target, dtype=np.float64)
    y_pred   = np.asarray(y_pred, dtype=np.float64)
    w        = np.asarray(w, dtype=np.float64)

    denom = np.sum(w * y_target ** 2)
    if denom == 0:
        return 0.0

    ratio   = np.sum(w * (y_target - y_pred) ** 2) / denom
    clipped = _clip01(ratio)
    val     = 1.0 - clipped

    return float(np.sqrt(val))


def evaluate_per_horizon(df, pred_col="prediction", target_col="y_target",
                         weight_col="weight", horizon_col="horizon"):
    scores = {}
    for h in sorted(df[horizon_col].unique()):
        mask = df[horizon_col] == h
        score = weighted_rmse_score(
            df.loc[mask, target_col],
            df.loc[mask, pred_col],
            df.loc[mask, weight_col],
        )
        scores[h] = score
        print(f"  Horizon {h:2d}: {score:.5f}")

    overall = weighted_rmse_score(
        df[target_col], df[pred_col], df[weight_col]
    )
    scores["overall"] = overall
    print(f"  {'Overall':>10s}: {overall:.5f}")

    return scores


# ========================================
# MAIN SCRIPT (KAGGLE SUBMISSION STYLE)
# ========================================

if __name__ == "__main__":
    # ── Run full pipeline ──
    sub, sub_raw, scores = train_and_predict_all_horizons()

    # ── Save submission ──
    import os
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    out_path = os.path.join(OUTPUT_DIR, "submission.csv")
    sub[["id", "prediction"]].to_csv(out_path, index=False)
    print(f"\nSaved: {out_path} ({len(sub):,} rows)")
    print(sub.head())

    # ── Print per-horizon scores ──
    print("\nVal scores per horizon:")
    for h, s in sorted(scores.items()):
        print(f"  Horizon {h:2d}: {s:.6f}")

    print("\nKaggle Submission Ready! 🚀")

